# Analisis de conjunto de datos

## Preparación de Datos

### Importación de librerías y descarga automática de Dataset

In [30]:
import os
import requests
import pandas as pd
import plotly.express as px
import numpy as np

def descargar_dataset():
    """
    Descarga el dataset de logs_exp_us.csv desde una URL pública 
    y lo guarda en la carpeta 'data', si no existe.
    """
    url = "https://code.s3.yandex.net/datasets/vehicles_us.csv"  # Reemplaza con la URL real
    archivo = 'data/vehicles_us.csv'

    # Verifica si el archivo ya existe
    if os.path.exists(archivo):
        print(f"✅ El archivo {archivo} ya existe")
        return True
    else:
        # Descargar el archivo
        print(f"📥 Descargando el dataset desde {url}...")
        try:
            response = requests.get(url)
            response.raise_for_status()  # Verifica que la descarga fue exitosa

            # Asegura que la carpeta 'data' exista
            os.makedirs('data', exist_ok=True)

            # Guarda el archivo descargado
            with open(archivo, 'wb') as f:
                f.write(response.content)

            # Verifica que se descargó correctamente        if os.path.exists(archivo):
                file_size = os.path.getsize(archivo) / (1024*1024)  # Tamaño en MB
                print(f"✅ Dataset descargado y guardado en {archivo} ({file_size:.2f} MB)")
            
            return True
        
        except requests.exceptions.RequestException as e:
            print(f"❌ ERROR al descargar el dataset: {e}")
            return False
        except Exception as e:
            print(f"❌ ERROR inesperado: {e}")
            return False
    
# Ejecutar la función de descarga
if descargar_dataset():
        print("¡Listo para comenzar el análisis!")

✅ El archivo data/vehicles_us.csv ya existe
¡Listo para comenzar el análisis!


### Carga de datos
Importamos las librerías necesarias para el análisis de datos y cargamos el archivo CSV. <br>
Se utilizan pandas para manejar la información en formato de tablas (DataFrames), numpy para realizar operaciones numéricas y matplotlib para crear gráficos. <br>
Se usa la función `pd.read_csv()` de pandas para leer el archivo y almacenarlo en un DataFrame.

In [31]:
df = pd.read_csv("./data/vehicles_us.csv")

df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed
0,9400,2011.0,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1.0,2018-06-23,19
1,25500,NaN,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1.0,2018-10-19,50
2,5500,2013.0,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,NaN,2019-02-07,79
3,1500,2003.0,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,NaN,2019-03-22,9
4,14900,2017.0,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,NaN,2019-04-02,28


## Análisis Exploratorio de Datos
En esta celda se utiliza el método `info()` para obtener un resumen del DataFrame, el número de filas, columnas, los tipos de datos y se muestra con el metodo `df.isna().sum()` cuántos valores nulos hay por columna.

In [32]:
# Información general del DataFrame
print('Información del DataFrame:\n')
df.info()

# Conteo de valores nulos
print('\nConteo de valores nulos por columna:')
df.isna().sum().sort_values(ascending=False).head(15)

Información del DataFrame:

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   price         51525 non-null  int64  
 1   model_year    47906 non-null  float64
 2   model         51525 non-null  object 
 3   condition     51525 non-null  object 
 4   cylinders     46265 non-null  float64
 5   fuel          51525 non-null  object 
 6   odometer      43633 non-null  float64
 7   transmission  51525 non-null  object 
 8   type          51525 non-null  object 
 9   paint_color   42258 non-null  object 
 10  is_4wd        25572 non-null  float64
 11  date_posted   51525 non-null  object 
 12  days_listed   51525 non-null  int64  
dtypes: float64(4), int64(2), object(7)
memory usage: 5.1+ MB

Conteo de valores nulos por columna:


is_4wd          25953
paint_color      9267
odometer         7892
cylinders        5260
model_year       3619
condition           0
model               0
price               0
fuel                0
type                0
transmission        0
date_posted         0
days_listed         0
dtype: int64

### Primer vistazo
Al dar un primer vistazo a nuestra base de datos, notamos que contiene miles de anuncios de vehículos en venta. Sin embargo, como ocurre en la vida real, los datos no son perfectos: faltan detalles en algunos anuncios, como el color del auto, el kilometraje o si tiene tracción en las 4 ruedas. Antes de sacar conclusiones, necesitamos limpiar el conjunto de datos.

### Comprobar elementos duplicados

In [47]:
# Número de duplicados
print(f"Duplicados antes: {df.duplicated().sum()}")

# Eliminación de duplicados
df = df.drop_duplicates().reset_index(drop=True)

print(f"Duplicados después: {df.duplicated().sum()}")

Duplicados antes: 0
Duplicados después: 0


### Valores únicos

In [48]:
for col in df.columns:
    print(f"{col}: {df[col].nunique()} valores únicos")

price: 3443 valores únicos
model_year: 68 valores únicos
model: 100 valores únicos
condition: 6 valores únicos
cylinders: 7 valores únicos
fuel: 5 valores únicos
odometer: 17762 valores únicos
transmission: 3 valores únicos
type: 13 valores únicos
paint_color: 12 valores únicos
is_4wd: 2 valores únicos
date_posted: 354 valores únicos
days_listed: 227 valores únicos


### Comprobar valores inusuales

In [11]:
print(df.describe())

               price    model_year     cylinders       odometer   is_4wd  \
count   51525.000000  47906.000000  46265.000000   43633.000000  25572.0   
mean    12132.464920   2009.750470      6.125235  115553.461738      1.0   
std     10040.803015      6.282065      1.660360   65094.611341      0.0   
min         1.000000   1908.000000      3.000000       0.000000      1.0   
25%      5000.000000   2006.000000      4.000000   70000.000000      1.0   
50%      9000.000000   2011.000000      6.000000  113000.000000      1.0   
75%     16839.000000   2014.000000      8.000000  155000.000000      1.0   
max    375000.000000   2019.000000     12.000000  990000.000000      1.0   

       days_listed  
count  51525.00000  
mean      39.55476  
std       28.20427  
min        0.00000  
25%       19.00000  
50%       33.00000  
75%       53.00000  
max      271.00000  


### Reparar la información incompleta
Para no descartar anuncios valiosos solo porque el vendedor olvidó un detalle, aplicamos reglas lógicas para rellenar esos espacios vacíos. <br>
Por ejemplo, si el anuncio no especifica que el auto es 4x4, asumimos que no lo es; y para detalles como el kilometraje o los cilindros, usamos promedios basados en autos similares. <br> Así mantenemos la integridad de nuestro análisis.

### Dando el formato correcto

In [55]:
# Convertir 'is_4wd' a entero (0 o 1)
df['is_4wd'] = df['is_4wd'].fillna(0).astype(int)

# Convertir fechas
df['date_posted'] = pd.to_datetime(df['date_posted'], errors='coerce')

# Asegurar que model_year sea entero
df['model_year'] = pd.to_numeric(df['model_year'], errors='coerce').astype('Int64')

Las computadoras son estrictas. Un año de fabricación registrado como "2015.0" ocupa más memoria y es más difícil de leer que un simple "2015". <br>En el paso anterior estandarizamos el formato de todos los números para que sean enteros, lo que hará que nuestros futuros gráficos sean mucho más claros.

### Calculando edad real del vehículo

In [62]:
# Convierte la columna de texto a un formato oficial de "Fecha" en Pandas
df['date_posted'] = pd.to_datetime(df['date_posted'], format='%Y-%m-%d')

# Extrae solo el año en que se publicó y resta el año del modelo
df['vehicle_age'] = df['date_posted'].dt.year - df['model_year']

# Sumamos 1 a las edades de "0"  y evitamos errores matemáticos más adelante (como divisiones entre cero)
df['vehicle_age'] = df['vehicle_age'] + 1

# Verifica que se haya creado la nueva columna
df[['date_posted', 'model_year', 'vehicle_age']].head()

,date_posted,model_year,vehicle_age
0,2018-06-23,2011,8
1,2018-10-19,<NA>,<NA>
2,2019-02-07,2013,7
3,2019-03-22,2003,17
4,2019-04-02,2017,3


El año de fabricación está bien, pero para entender verdaderamente el valor de un auto, es más útil saber "cuántos años de uso tiene". <br> Aquí creamos una nueva columna que calcula la edad exacta del vehículo basándonos en el año actual (2024).

#### Confirmación de cambios del formato

In [63]:
# Tabla de datos después de la limpieza inicial
df.head()

,price,model_year,model,condition,cylinders,fuel,odometer,transmission,type,paint_color,is_4wd,date_posted,days_listed,vehicle_age
0,9400,2011,bmw x5,good,6.0,gas,145000.0,automatic,SUV,NaN,1,2018-06-23,19,8
1,25500,<NA>,ford f-150,good,6.0,gas,88705.0,automatic,pickup,white,1,2018-10-19,50,<NA>
2,5500,2013,hyundai sonata,like new,4.0,gas,110000.0,automatic,sedan,red,0,2019-02-07,79,7
3,1500,2003,ford f-150,fair,8.0,gas,NaN,automatic,pickup,NaN,0,2019-03-22,9,17
4,14900,2017,chrysler 200,excellent,4.0,gas,80903.0,automatic,sedan,black,0,2019-04-02,28,3


In [59]:
# Información de datos después de la limpieza
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51525 entries, 0 to 51524
Data columns (total 13 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         51525 non-null  int64         
 1   model_year    47906 non-null  Int64         
 2   model         51525 non-null  object        
 3   condition     51525 non-null  object        
 4   cylinders     46265 non-null  float64       
 5   fuel          51525 non-null  object        
 6   odometer      43633 non-null  float64       
 7   transmission  51525 non-null  object        
 8   type          51525 non-null  object        
 9   paint_color   42258 non-null  object        
 10  is_4wd        51525 non-null  int64         
 11  date_posted   51525 non-null  datetime64[ns]
 12  days_listed   51525 non-null  int64         
dtypes: Int64(1), datetime64[ns](1), float64(2), int64(3), object(6)
memory usage: 5.2+ MB


### Aplicación de filtros
* El precio no puede ser cero o negativo.
* El odometro no puede ser negativo.
* Año fuera de 1900–2025 se considera inválido.

In [67]:
def calcular_limites(df, columna, min_valor=0):
    """
    Calcula los límites inferior y superior usando el Rango Intercuartílico (IQR).
    Evita que el límite inferior sea menor a 'min_valor'.
    """
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    
    lim_inf = max(min_valor, Q1 - 1.5 * IQR)
    lim_sup = Q3 + 1.5 * IQR
    
    return lim_inf, lim_sup

# Aplicamos la función a nuestras 3 columnas clave
lim_inf_price, lim_sup_price = calcular_limites(df, 'price', min_valor=1)
lim_inf_age, lim_sup_age = calcular_limites(df, 'vehicle_age', min_valor=0)
lim_inf_odo, lim_sup_odo = calcular_limites(df, 'odometer', min_valor=0)

# Imprimir para verificar que no sean números locos
print(f"Límites Edad: {lim_inf_age} a {lim_sup_age} años")
print(f"Límites Kilometraje: {lim_inf_odo} a {lim_sup_odo} millas/km")
print(f"Límites Precio: ${lim_inf_price} a ${lim_sup_price}")

Límites Edad: 0 a 25.0 años
Límites Kilometraje: 0 a 282500.0 millas/km
Límites Precio: $1 a $34597.5


In [68]:
df_clean = df[(df['price'] > 0) & (df['odometer'] >= 0)]
df_clean = df_clean[df_clean['model_year'].between(1900, 2025)].copy()

In [69]:
# Filtros de valores fuera de rango

df_clean = df[(df['price'] > lim_inf_price) & (df['price'] < lim_sup_price) &
              (df['vehicle_age'] > lim_inf_age) & (df['vehicle_age'] < lim_sup_age) &
              (df['odometer'] > lim_inf_odo) & (df['odometer'] < lim_sup_odo)].copy()

# Confirmar cambios
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 37319 entries, 0 to 51523
Data columns (total 14 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   price         37319 non-null  int64         
 1   model_year    37319 non-null  Int64         
 2   model         37319 non-null  object        
 3   condition     37319 non-null  object        
 4   cylinders     33506 non-null  float64       
 5   fuel          37319 non-null  object        
 6   odometer      37319 non-null  float64       
 7   transmission  37319 non-null  object        
 8   type          37319 non-null  object        
 9   paint_color   30643 non-null  object        
 10  is_4wd        37319 non-null  int64         
 11  date_posted   37319 non-null  datetime64[ns]
 12  days_listed   37319 non-null  int64         
 13  vehicle_age   37319 non-null  Int64         
dtypes: Int64(2), datetime64[ns](1), float64(2), int64(3), object(6)
memory usage: 4.3+ MB


### Eliminando ruido y precios irreales
En internet es común encontrar errores de captura: autos listados a 1 dólar o vehículos de hace 100 años a precios exorbitantes. Si incluimos esas exageraciones (conocidas como valores atípicos), nuestras gráficas y promedios se verían completamente distorsionados. Aquí aplicamos un filtro matemático para quedarnos solo con el mercado real, eliminando el 10% de los datos extremos en precio, edad y kilometraje.

In [70]:
# Guardamos el dataset limpio en la carpeta 'data'
ruta_limpia = 'data/vehicles_us_clean.csv'
df_clean.to_csv(ruta_limpia, index=False)

print(f"✅ ¡Éxito! Dataset limpio guardado en: {ruta_limpia}")

✅ ¡Éxito! Dataset limpio guardado en: data/vehicles_us_clean.csv


### Base de datos lista para tomar decisiones
¡Misión cumplida! Hemos transformado una tabla desordenada en una fuente de información confiable y pulida. Hemos guardado esta nueva versión limpia, la cual será nuestra base para crear los gráficos interactivos y descubrir qué hace que un auto se venda más rápido o más caro.

In [79]:
print("🚗 Total de autos ANTES del filtro:", len(df))
print("🧼 Total de autos DESPUÉS del filtro:", len(df_clean))
print("💰 Precio MÁXIMO en el dataset limpio:", df_clean['price'].max())
print("🛑 Límite superior que se calculo:", lim_sup_price)

🚗 Total de autos ANTES del filtro: 51525
🧼 Total de autos DESPUÉS del filtro: 37319
💰 Precio MÁXIMO en el dataset limpio: 34595
🛑 Límite superior que se calculo: 34597.5


### Distribución de precio

In [80]:
fig_price = px.histogram(
    df_clean, 
    x="price", 
    nbins=60, 
    title="Distribución de precios de los vehículos",
    labels={"price": "Precio (USD)"}
)
fig_price.show()

### Kilometraje

In [81]:
fig_odo = px.histogram(
    df_clean, 
    x="odometer", 
    nbins=60, 
    title="Distribución de kilometraje (odometer)",
    labels={"odometer": "Kilometraje"}
)
fig_odo.show()

### Relación entre precio, kilometraje y año del vehículo 

In [82]:
df_fixed = df_clean.copy()
df_fixed["cylinders"] = df_fixed["cylinders"].fillna(4)

fig_scatter = px.scatter(
    df_fixed.sample(3000, random_state=42),
    x="odometer",
    y="price",
    color="model_year",            # Escala de color continua por año del vehículo
    size="cylinders",              # Tamaño de burbuja según número de cilindros
    hover_data=["model", "condition", "fuel", "transmission"],
    title="Relación entre precio, kilometraje y año del vehículo",
    labels={
        "odometer": "Kilometraje (mi)",
        "price": "Precio (USD)",
        "model_year": "Año del vehículo",
        "cylinders": "Cilindros"
    },
    color_continuous_scale="Viridis",  # Escala de color agradable y perceptiva
    size_max=15,                       # Tamaño máximo de las burbujas
    opacity=0.7                        # Transparencia para evitar saturación
)

fig_scatter.update_layout(
    plot_bgcolor="white",
    xaxis=dict(showgrid=True, gridcolor="lightgray"),
    yaxis=dict(showgrid=True, gridcolor="lightgray"),
    title_x=0.5
)

fig_scatter.show()